# Weekly project

Today you are going to implement the last parts of the algorithm you started on monday. For reference you can see it below.

![title](algorithm_3.png)

It is a good idea to follow and track the steps in the algorithm in the below implementation. Only take one step at a time.

Once you have the algorithm up and running you can try with a larger dataset to see if your algorithm is able to maintain good accurracy over a longer distance. The larger dataset can be found here:
[Left images](https://dtudk-my.sharepoint.com/:u:/g/personal/evanb_dtu_dk/EQu8kmGBDDROtGJ7IkZB2tQBJrxmgY9t8LVM_JuEi83TYw)
[Right images](https://dtudk-my.sharepoint.com/:u:/g/personal/evanb_dtu_dk/EcKI_zrXTvpMulizidCZm4oBLJcQ_LTV9Zs6oQFF74JTRQ)

In [3]:
import numpy as np
import cv2 as cv2
import glob

def getK():
    return np.array([[7.188560e+02, 0.000000e+00, 6.071928e+02],
                     [0,            7.188560e+02, 1.852157e+02],
                     [0,            0,             1]])

def getTruePose():
    file = '00.txt'
    return np.genfromtxt(file, delimiter=' ', dtype=None)

<details>
  <summary>tips:</summary>

- [Feature Matching](https://docs.opencv.org/4.x/dc/dc3/tutorial_py_matcher.html)
- To get the keypoint of a certain match do: ```kp1[match.queryIdx].pt``` and ```kp2[match.trainIdx].pt```

In [4]:
left_img_paths = sorted(glob.glob('left/*.png'))
right_img_paths = sorted(glob.glob('right/*.png'))
baseline = 0.54
K = getK()

left_img = cv2.imread(left_img_paths[0], 0)
right_img = cv2.imread(right_img_paths[0], 0)

extract_keypoints_sift(left_img, right_img, K, baseline)

NameError: name 'extract_keypoints_sift' is not defined

In [5]:
def extract_keypoints_sift(img1, img2, K, baseline):
    """
    Use SIFT to detect keypoints and compute descriptors in both images,
    and find matches between them using knnMatch with k=2
    """
    sift = cv2.SIFT_create()
    kp1, desc1 = sift.detectAndCompute(img1, None)
    kp2, desc2 = sift.detectAndCompute(img2, None)

    matcher = cv2.DescriptorMatcher_create(cv2.DescriptorMatcher_FLANNBASED)
    knn_matches = matcher.knnMatch(desc1, desc2, 2)
    """
    Get the best keypoints by applying the Lowes ratio test
    """
    # print("\nkpq", kp1)
    #print("\nknn_matches",knn_matches)
    ratio_thresh = 0.8
    good_matches = []
    for m,n in knn_matches:
        if m.distance < ratio_thresh * n.distance:
            good_matches.append(m)
    # print("\ngood_matches",good_matches)
    match_points1 = [kp1[match.queryIdx].pt for match in good_matches]
    match_points2 = [kp2[match.trainIdx].pt for match in good_matches]
    # Transform best keypoints to nparray
    p1 = np.array(match_points1).astype('float32')
    p2 = np.array(match_points2).astype('float32')
    

    ##### ############# ##########
    ##### Do Triangulation #######
    ##### ########################
    # project the feature points to 3D with triangulation
    # projection matrix for Left and Right Image
    M_left = K.dot(np.hstack((np.eye(3), np.zeros((3, 1)))))
    M_rght = K.dot(np.hstack((np.eye(3), np.array([[-baseline, 0, 0]]).T)))

    p1_flip = np.vstack((p1.T, np.ones((1, p1.shape[0]))))
    p2_flip = np.vstack((p2.T, np.ones((1, p2.shape[0]))))

    P = cv2.triangulatePoints(M_left, M_rght, p1_flip[:2], p2_flip[:2])

    # Normalize homogeneous coordinates (P->Nx4  [N,4] is the normalizer/scale)
    P = P / P[3]
    land_points = P[:3]

    return land_points.T, p1

In [6]:
left_img_paths = sorted(glob.glob('left/*.png'))
right_img_paths = sorted(glob.glob('right/*.png'))
baseline = 0.54
K = getK()

left_img = cv2.imread(left_img_paths[0], 0)
right_img = cv2.imread(right_img_paths[0], 0)

extract_keypoints_sift(left_img, right_img, K, baseline)

(array([[ 0.40340564,  0.06257033, -0.48355554],
        [ 0.81117063,  0.08179962, -0.97349791],
        [ 0.42229251,  0.07739688, -0.50878162],
        ...,
        [ 3.49392126,  0.86441859,  4.14729074],
        [ 3.45830852, -0.88144306,  4.07231213],
        [ 3.50182787,  0.1092062 ,  4.103894  ]], shape=(1119, 3)),
 array([[   7.491204,   93.41088 ],
        [   8.51517 ,  114.19539 ],
        [  13.867567,  114.440186],
        ...,
        [1212.799   ,  334.6209  ],
        [1217.6628  ,   30.017338],
        [1220.5883  ,  204.41551 ]], shape=(1119, 2), dtype=float32))

In [7]:
def featureTracking(prev_img, next_img, prev_points, world_points):
    """
    Use OpenCV to find the prev_points from the prev_img in the next_img
    Remember to remove points that could not be found from prev_points, next_points, and world_points
    hint: status == 1
    """
    params = dict(winSize=(21, 21),
                 maxLevel=3,
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
    
    #next_points, status, _ = ...
    next_points, status, err = cv2.calcOpticalFlowPyrLK(prev_img, next_img, prev_points, None, **params)
    good = status.ravel() == 1
    return world_points[good], prev_points[good], next_points[good]


In [8]:
#from importlib.resources.readers import remove_duplicates
def remove_duplicates(queryPoints, refPoints, radius=5):
    # remove duplicate points from new query points,
    for i in range(len(queryPoints)):
        query = queryPoints[i]
        xliml, xlimh = query[0] - radius, query[0] + radius
        yliml, ylimh = query[1] - radius, query[1] + radius
        inside_x_lim_mask = (refPoints[:, 0] > xliml) & (refPoints[:, 0] < xlimh)
        curr_kps_in_x_lim = refPoints[inside_x_lim_mask]

        if curr_kps_in_x_lim.shape[0] != 0:
            inside_y_lim_mask = (curr_kps_in_x_lim[:, 1] > yliml) & (curr_kps_in_x_lim[:, 1] < ylimh)
            curr_kps_in_x_lim_and_y_lim = curr_kps_in_x_lim[inside_y_lim_mask, :]
            if curr_kps_in_x_lim_and_y_lim.shape[0] != 0:
                queryPoints[i] = np.array([0, 0])
    return (queryPoints[:, 0] != 0)

def playImageSequence(l_img_paths, r_img_paths, K):
    baseline = 0.54
    left_img = cv2.imread(l_img_paths[0], 0)
    right_img = cv2.imread(r_img_paths[0], 0)

    ##### ################################# #######
    ##### Get 3D points Using Triangulation #######
    ##### #########################################
    """
    Implement step 1.2 and 1.3
    Store the features in 'reference_2D' and the 3D points (landmarks) in 'landmark_3D'
    hint: use 'extract_keypoints_sift' above
    """
    # Step 1.2 and 1.3 
    landmark_3D, reference_2D = extract_keypoints_sift(left_img, right_img, K, baseline)
    #print("land points", landmark_3D, "p1", reference_2D)

    # Groundtruth for plot
    truePose = getTruePose()
    traj = np.zeros((600, 600, 3), dtype=np.uint8)
    maxError = 0

    reference_img = left_img

    for i in range(1, 101):
        print('image: ', i)
        curImage = cv2.imread(l_img_paths[i], 0)
        curImage_R = cv2.imread(r_img_paths[i], 0)

        ##### ############################################################# #######
        ##### Calculate 2D and 3D feature correspndances in t=T-1, and t=T  #######
        ##### #####################################################################
        """
        Implement step 2.2)
        Remember this is a part of a loop, so the initial features are already
        provided in step 1)-1.3) outside the loop in 'reference_2D' and 'landmark_3D'
        """
        landmark_3D, reference_2D, next_points = featureTracking(reference_img, curImage, reference_2D, landmark_3D)

        ##### ####################################### #######
        ##### Calculate relative pose using PNPRansac #######
        ##### ###############################################
        """
        Implement step 2.3)
        """
        distortion_coeffs = np.array([0,0,0,0])
        _, rvec, tvec, inliers = cv2.solvePnPRansac(landmark_3D, next_points, K, distortion_coeffs)

        ##### ####################################################### #######
        ##### Get Pose and Tranformation Matrix in world coordionates #######
        ##### ###############################################################
        rot, _ = cv2.Rodrigues(rvec)
        tvec = -rot.T.dot(tvec)  # coordinate transformation, from camera to world. What is the XYZ of the camera wrt World
        inv_transform = np.hstack((rot.T, tvec))  # inverse transform. A tranform projecting points from the camera frame to the world frame

        ##### ################################# #######
        ##### Get 3D points Using Triangulation #######
        ##### #########################################
        # re-obtain the 3D points. 
        # hint: use 'extract_keypoints_sift' agaitrajn
        """
        Implement step 2.4)
        """
        landmark_3D_new, reference_2D_new = extract_keypoints_sift(curImage, curImage_R, K, baseline)

        reference_2D_new = reference_2D_new.astype('float32')

        # Hint 2: drop new points behind the camera (z <= 0 in the camera frame)
        in_front = landmark_3D_new[:, 2] > 0
        landmark_3D_new = landmark_3D_new[in_front]
        reference_2D_new = reference_2D_new[in_front]

        # Project the points from camera to world coordinates
        landmark_3D_new = inv_transform.dot(
            np.vstack((landmark_3D_new.T, np.ones((1, landmark_3D_new.shape[0]))))).T

        # Keep only new 2D points that are not within 5 px of an already-tracked point
        keep = remove_duplicates(reference_2D_new.copy(), next_points)

        # Combine old tracked points (at their position in curImage) with the new ones
        reference_2D = np.vstack((next_points, reference_2D_new[keep])).astype('float32')
        landmark_3D = np.vstack((landmark_3D, landmark_3D_new[keep]))

        ##### ####################### #######
        ##### Done, Next image please #######
        ##### ###############################
        reference_img = curImage

        ##### ################################## #######
        ##### START OF Print and visualize stuff #######
        ##### ##########################################
        # draw images
        draw_x, draw_y = int(tvec[0,0]) + 300, 600-(int(tvec[2, 0]) + 100)
        true_x, true_y = int(truePose[i][3]) + 300, 600-(int(truePose[i][11]) + 100)

        curError = float(np.sqrt(
            (tvec[0, 0] - truePose[i][3]) ** 2 +
            (tvec[1, 0] - truePose[i][7]) ** 2 +
            (tvec[2, 0] - truePose[i][11]) ** 2))
        
        if (curError > maxError):
            maxError = curError

        print(tvec[0],tvec[1],tvec[2], rvec[0], rvec[1], rvec[2])
        print([truePose[i][3], truePose[i][7], truePose[i][11]])
        
        text = "Coordinates: x ={0:02f}m y = {1:02f}m z = {2:02f}m".format(float(tvec[0, 0]), float(tvec[1, 0]),
                                                                           float(tvec[2, 0]))
        cv2.circle(traj, (draw_x, draw_y), 1, (0, 0, 255), 2)
        cv2.circle(traj, (true_x, true_y), 1, (255, 0, 0), 2)
        cv2.rectangle(traj, (10, 30), (550, 50), (0, 0, 0), cv2.FILLED)
        cv2.putText(traj, text, (10, 50), cv2.FONT_HERSHEY_PLAIN, 1, (255, 255, 255), 1, 8)

        h1, w1 = traj.shape[:2]
        h2, w2 = curImage.shape[:2]
        vis = np.zeros((max(h1, h2), w1 + w2, 3), np.uint8)
        vis[:h1, :w1, :3] = traj
        vis[:h2, w1:w1 + w2, :3] = np.dstack((np.dstack((curImage,curImage)),curImage))

        cv2.imshow("Trajectory", vis)
        k = cv2.waitKey(1) & 0xFF
        if k == 27: break


    cv2.waitKey(0)
    cv2.destroyAllWindows()
    print('Maximum Error: ', maxError)
    ##### ################################ #######
    ##### END OF Print and visualize stuff #######
    ##### ########################################

In [9]:
# Load image paths
left_img_paths = sorted(glob.glob('left/*.png'))
right_img_paths = sorted(glob.glob('right/*.png'))

K = getK()

playImageSequence(left_img_paths, right_img_paths, K)

image:  1
[-0.00208477] [-0.00607844] [0.67441707] [-0.00226396] [0.00335153] [-0.00249753]
[np.float64(-0.04690294), np.float64(-0.02839928), np.float64(0.8586941)]


QFontDatabase: Cannot find font directory /home/madsac/miniconda3/envs/pfas/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/madsac/miniconda3/envs/pfas/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/madsac/miniconda3/envs/pfas/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/madsac/miniconda3/envs/pfas/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find f

image:  2
[-0.01128802] [-0.01303202] [1.37109039] [-0.00376865] [0.00759562] [-0.00093634]
[np.float64(-0.09374345), np.float64(-0.05676064), np.float64(1.716275)]
image:  3
[-0.02833471] [-0.01809693] [2.08624359] [-0.00523077] [0.01160318] [-0.00090186]
[np.float64(-0.1406429), np.float64(-0.08515762), np.float64(2.574964)]
image:  4
[-0.05224635] [-0.02575325] [2.81497986] [-0.00588054] [0.01624747] [0.00051329]
[np.float64(-0.1874858), np.float64(-0.1135202), np.float64(3.432648)]
image:  5
[-0.06981232] [-0.03480548] [3.57229752] [-0.00614453] [0.02078059] [-0.00056065]
[np.float64(-0.2343818), np.float64(-0.141915), np.float64(4.291335)]
image:  6
[-0.09243278] [-0.05008865] [4.31400287] [-0.00722643] [0.02548424] [0.0011649]
[np.float64(-0.2812195), np.float64(-0.1702743), np.float64(5.148987)]
image:  7
[-0.12455343] [-0.0562422] [5.10200656] [-0.00801807] [0.03066747] [0.00406323]
[np.float64(-0.3281178), np.float64(-0.1986703), np.float64(6.007777)]
image:  8
[-0.16199222] [

# Challenge 
The current implementation only uses features computed at the current timestep. However, as we process more images we potentially have a lot of features from previous timesteps that are still valid. The challenge is to expand the `extract_keypoints_surf(..., refPoints)` function by giving it old reference points. You should then combine your freshly computed features with the old features and remove all duplicates. This requires you to keep track of old features and 3D points.

Hint 1: The following function `removeDuplicate` can be used for removing duplicates.

Hint 2: you are not interested in points that are behind you, so remember to remove points that are negative in the direction you move.